In [1]:
from pathlib import Path
from itertools import zip_longest

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Input files
CANDIDATE_PATH = Path("five_metal_compounds_permutation.csv")
FEATURE_PATH = Path("data/five_metal_compounds_permutation_features.csv")

# Output files
OUTPUT_DIR = Path("results/phi_mis_screening")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_OUTPUT = OUTPUT_DIR / "all_five_metal_phi_mis.csv"
SELECTED_OUTPUT = OUTPUT_DIR / "five_metal_phi_mis_0_to_5.csv"

ALL_TEMP = OUTPUT_DIR / "all_five_metal_phi_mis.tmp"
SELECTED_TEMP = OUTPUT_DIR / "five_metal_phi_mis_0_to_5.tmp"

CHUNK_SIZE = 20_000
EXPECTED_ROWS = 4_088_700

for path in [CANDIDATE_PATH, FEATURE_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

candidate_reader = pd.read_csv(
    CANDIDATE_PATH,
    usecols=["base", "compound"],
    dtype={"base": "string", "compound": "string"},
    chunksize=CHUNK_SIZE,
)

feature_reader = pd.read_csv(
    FEATURE_PATH,
    usecols=[
        "compound",
        "M_EN_mismatch",
        "M_FIE_mismatch",
        "M_AR_mismatch",
    ],
    dtype={"compound": "string"},
    chunksize=CHUNK_SIZE,
)

total_count = 0
selected_count = 0
undefined_count = 0

with ALL_TEMP.open("w", encoding="utf-8", newline="") as all_file, \
     SELECTED_TEMP.open("w", encoding="utf-8", newline="") as selected_file:
    
    for chunk_index, (candidates, features) in enumerate(
        tqdm(
            zip_longest(candidate_reader, feature_reader),
            desc="Calculating Phi_mis",
        )
    ):
        if candidates is None or features is None:
            raise ValueError(
                "The candidate and feature files have different row counts."
            )

        candidates = candidates.reset_index(drop=True)
        features = features.reset_index(drop=True)

        if not candidates["compound"].equals(features["compound"]):
            mismatch = candidates["compound"].ne(
                features["compound"]
            ).fillna(True)

            first_mismatch = mismatch[mismatch].index[0]

            raise ValueError(
                f"Compound mismatch in chunk {chunk_index}, "
                f"row {first_mismatch}. "
                f"Candidate: {candidates.at[first_mismatch, 'compound']}; "
                f"feature: {features.at[first_mismatch, 'compound']}"
            )

        result = candidates.copy()

        for column in [
            "M_EN_mismatch",
            "M_FIE_mismatch",
            "M_AR_mismatch",
        ]:
            result[column] = pd.to_numeric(
                features[column],
                errors="coerce",
            )

        numerator = (
            result["M_EN_mismatch"]
            + result["M_FIE_mismatch"]
        )

        denominator = result["M_AR_mismatch"].replace(0, np.nan)

        result["Phi_mis"] = (
            numerator / denominator
        ).replace([np.inf, -np.inf], np.nan)

        selected = result.loc[
            result["Phi_mis"].between(
                0,
                5,
                inclusive="both",
            )
        ]

        result.to_csv(
            all_file,
            index=False,
            header=(chunk_index == 0),
        )

        selected.to_csv(
            selected_file,
            index=False,
            header=(chunk_index == 0),
        )

        total_count += len(result)
        selected_count += len(selected)
        undefined_count += int(result["Phi_mis"].isna().sum())

if total_count != EXPECTED_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_ROWS:,} compounds, "
        f"but processed {total_count:,}. "
        "The temporary files were not published."
    )

ALL_TEMP.replace(ALL_OUTPUT)
SELECTED_TEMP.replace(SELECTED_OUTPUT)

print("Total compounds:", f"{total_count:,}")
print("Compounds with 0 <= Phi_mis <= 5:", f"{selected_count:,}")
print("Undefined Phi_mis:", f"{undefined_count:,}")
print("All results:", ALL_OUTPUT.resolve())
print("Selected results:", SELECTED_OUTPUT.resolve())

Calculating Phi_mis: 0it [00:00, ?it/s]

Total compounds: 4,088,700
Compounds with 0 <= Phi_mis <= 5: 1,318,640
Undefined Phi_mis: 0
All results: D:\code\five-metal\results\phi_mis_screening\all_five_metal_phi_mis.csv
Selected results: D:\code\five-metal\results\phi_mis_screening\five_metal_phi_mis_0_to_5.csv
